In [11]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import algoritmos_clasificacion as t
import importlib
importlib.reload(t)

<module 'algoritmos_clasificacion' from '/home/hezek/FING/MachineLearning/MachineLearningLab/LAB1/Script/algoritmos_clasificacion.py'>

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.metrics import (
    make_scorer, f1_score, recall_score, precision_score, accuracy_score, confusion_matrix
)
import algoritmos_clasificacion as t
import numpy as np
import pandas as pd

# Dataset
S = t.cargarDataset("data_dia")
X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto",
    "dir_viento_carrasco_prom","int_viento_carrasco_prom",
    "int_racha_viento_carrasco_prom","dir_viento_salto_prom","int_viento_salto_prom",
    "int_racha_viento_salto_prom","vis_carrasco_prom","vis_salto_prom","fen_carrasco_17", "fase_lunar"]]

# X = S[["hum_carrasco","tmax_carrasco","tmin_carrasco",
#        "amplitud_carrasco","dir_viento_carrasco_prom","int_viento_carrasco_prom",
#        "int_racha_viento_carrasco_prom","vis_carrasco_prom","fen_carrasco_17", "fase_lunar"]]
y = S["dia_lluvioso"]

# 1. Train/Test split (80/20 estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=20
)

# 2. Definir validación cruzada en TRAIN
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)

criterios = ["gain", "gini", "gain_ratio"]
min_info_gains = [0, 0.004,0.006, 0.008]

# Estructura para guardar resultados
results = []

# Definir scoring (incluye varias métricas, pero se seleccionará F1 como principal)
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score, pos_label='si'),
    'recall': make_scorer(recall_score, pos_label='si'),
    'precision': make_scorer(precision_score, pos_label='si')
}

print("=== EVALUACIÓN DE TODAS LAS COMBINACIONES (en train con CV) ===\n")

for i in criterios:
    for j in min_info_gains:
        print(f"--- Evaluando: Criterio={i}, Min_Info_Gain={j} ---")
        model = t.ID3Classifier(min_info_gain=j, criterio=i)
        cv_results = cross_validate(model, X_train, y_train, cv=kf, scoring=scoring)

        # Calcular promedios y desviaciones
        row = {
            'criterio': i,
            'min_info_gain': j,
            'accuracy_mean': np.mean(cv_results['test_accuracy']),
            'accuracy_std': np.std(cv_results['test_accuracy']),
            'f1_mean': np.mean(cv_results['test_f1']),
            'f1_std': np.std(cv_results['test_f1']),
            'recall_mean': np.mean(cv_results['test_recall']),
            'recall_std': np.std(cv_results['test_recall']),
            'precision_mean': np.mean(cv_results['test_precision']),
            'precision_std': np.std(cv_results['test_precision'])
        }
        results.append(row)

df_results = pd.DataFrame(results)

print("\nRESUMEN COMPLETO DE RESULTADOS (CV en train)")
print("================================")
print(df_results.round(4).to_string(index=False))

# 3. Seleccionar la mejor combinación según F1
best_row_F = df_results.loc[df_results['f1_mean'].idxmax()]
print("\nMEJOR CONFIGURACIÓN EN TRAIN (según F1):")
#print(best_row)
# 3.2 Seleccionar la mejor combinación según acc

best_row_A = df_results.loc[df_results['accuracy_mean'].idxmax()]
print("\nMEJOR CONFIGURACIÓN EN TRAIN (según accuracy):")
#print(best_row2)

# 4. Entrenar modelo final en TODO el train con la mejor config
best_model_f = t.ID3Classifier(
    min_info_gain=best_row_F['min_info_gain'],
    criterio=best_row_F['criterio']
)

best_model_a = t.ID3Classifier(
    min_info_gain=best_row_A['min_info_gain'],
    criterio=best_row_A['criterio']
)

best_model_f.fit(X_train, y_train)
best_model_a.fit(X_train, y_train)

# 5. Evaluar en TEST
y_pred_f = best_model_f.predict(X_test)
y_pred_a = best_model_a.predict(X_test)

print("\nEVALUACIÓN FINAL EN TEST según mejor criterio por f-score:")
print("mejor criterio: ", best_row_F['criterio'], "mejor min_info_gain: ", best_row_F['min_info_gain'])
print("Accuracy :", accuracy_score(y_test, y_pred_f))
print("F1       :", f1_score(y_test, y_pred_f, pos_label='si'))
print("Recall   :", recall_score(y_test, y_pred_f, pos_label='si'))
print("Precision:", precision_score(y_test, y_pred_f, pos_label='si'))

print("\nEVALUACIÓN FINAL EN TEST según mejor criterio por accuracy:")
print("mejor criterio: ", best_row_A['criterio'], "mejor min_info_gain: ", best_row_A['min_info_gain'])
print("Accuracy :", accuracy_score(y_test, y_pred_a))
print("F1       :", f1_score(y_test, y_pred_a, pos_label='si'))
print("Recall   :", recall_score(y_test, y_pred_a, pos_label='si'))
print("Precision:", precision_score(y_test, y_pred_a, pos_label='si'))

# 6. Matriz de confusión

print("\nMATRIZ DE CONFUSIÓN (TEST) según mejor criterio por f-score")
cm = confusion_matrix(y_test, y_pred_f, labels=['si', 'no'])
cm_df = pd.DataFrame(cm,
                     index=["Actual: si", "Actual: no"],
                     columns=["Pred: si", "Pred: no"])

print(cm_df)
print("===========================")

print("\nMATRIZ DE CONFUSIÓN (TEST) según mejor criterio por accuracy")
cm = confusion_matrix(y_test, y_pred_a, labels=['si', 'no'])
cm_df = pd.DataFrame(cm,
                     index=["Actual: si", "Actual: no"],
                     columns=["Pred: si", "Pred: no"])
print(cm_df)
print("===========================")



=== EVALUACIÓN DE TODAS LAS COMBINACIONES (en train con CV) ===

--- Evaluando: Criterio=gain, Min_Info_Gain=0 ---
--- Evaluando: Criterio=gain, Min_Info_Gain=0.004 ---
--- Evaluando: Criterio=gain, Min_Info_Gain=0.006 ---
--- Evaluando: Criterio=gain, Min_Info_Gain=0.008 ---
--- Evaluando: Criterio=gini, Min_Info_Gain=0 ---
--- Evaluando: Criterio=gini, Min_Info_Gain=0.004 ---
--- Evaluando: Criterio=gini, Min_Info_Gain=0.006 ---
--- Evaluando: Criterio=gini, Min_Info_Gain=0.008 ---
--- Evaluando: Criterio=gain_ratio, Min_Info_Gain=0 ---
--- Evaluando: Criterio=gain_ratio, Min_Info_Gain=0.004 ---
--- Evaluando: Criterio=gain_ratio, Min_Info_Gain=0.006 ---
--- Evaluando: Criterio=gain_ratio, Min_Info_Gain=0.008 ---

RESUMEN COMPLETO DE RESULTADOS (CV en train)
  criterio  min_info_gain  accuracy_mean  accuracy_std  f1_mean  f1_std  recall_mean  recall_std  precision_mean  precision_std
      gain          0.000         0.6947        0.0041   0.2605  0.0212       0.2433      0.0239     

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.metrics import (
    make_scorer, f1_score, recall_score, precision_score, accuracy_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd
import algoritmos_clasificacion as t

# 1. Cargar dataset
S = t.cargarDataset("data_dia_rf")

# 2. Atributos y variable objetivo
X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto",
       "amplitud_carrasco","amplitud_salto","dir_viento_carrasco_prom","int_viento_carrasco_prom",
       "int_racha_viento_carrasco_prom","dir_viento_salto_prom","int_viento_salto_prom",
       "int_racha_viento_salto_prom","vis_carrasco_prom","vis_salto_prom"]]
y = S["dia_lluvioso"]

# 3. Train/Test split (80/20 estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=20, stratify=y
)

# 4. Configurar validación cruzada
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)

# 5. Lista de estimadores y estructura para resultados
n_estimators_values = [100, 200, 500, 1000, 2000]
results = []

for n_est in n_estimators_values:
    print(f"\n=== Evaluando Random Forest con n_estimators={n_est} ===")
    
    rf_model = RandomForestClassifier(n_estimators=n_est, random_state=20)
    
    # Cross-validate en train
    scoring = {
        'accuracy': make_scorer(accuracy_score),
        'f1': make_scorer(f1_score, pos_label='si'),
        'recall': make_scorer(recall_score, pos_label='si'),
        'precision': make_scorer(precision_score, pos_label='si')
    }
    
    cv_results = cross_validate(rf_model, X_train, y_train, cv=kf, scoring=scoring)
    
    # Promedios y std
    row = {
        'n_estimators': n_est,
        'accuracy_mean': np.mean(cv_results['test_accuracy']),
        'accuracy_std': np.std(cv_results['test_accuracy']),
        'f1_mean': np.mean(cv_results['test_f1']),
        'f1_std': np.std(cv_results['test_f1']),
        'recall_mean': np.mean(cv_results['test_recall']),
        'recall_std': np.std(cv_results['test_recall']),
        'precision_mean': np.mean(cv_results['test_precision']),
        'precision_std': np.std(cv_results['test_precision'])
    }
    results.append(row)

# Convertir a DataFrame
df_results = pd.DataFrame(results)
print("\nRESUMEN COMPLETO RANDOM FOREST (CV en train)")
print("===========================================")
print(df_results.round(4).to_string(index=False))

# 6. Seleccionar la mejor configuración según F1 y Accuracy
best_row_F = df_results.loc[df_results['f1_mean'].idxmax()]
best_row_A = df_results.loc[df_results['accuracy_mean'].idxmax()]

# Entrenar modelos finales en todo train
best_model_f = RandomForestClassifier(n_estimators=best_row_F['n_estimators'], random_state=20)
best_model_a = RandomForestClassifier(n_estimators=best_row_A['n_estimators'], random_state=20)

best_model_f.fit(X_train, y_train)
best_model_a.fit(X_train, y_train)

# Evaluar en test
y_pred_f = best_model_f.predict(X_test)
y_pred_a = best_model_a.predict(X_test)

# Métricas test
def print_metrics(y_true, y_pred, label):
    print(f"\n=== Evaluación TEST: {label} ===")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("F1       :", f1_score(y_true, y_pred, pos_label='si'))
    print("Recall   :", recall_score(y_true, y_pred, pos_label='si'))
    print("Precision:", precision_score(y_true, y_pred, pos_label='si'))
    cm = confusion_matrix(y_true, y_pred, labels=['si','no'])
    cm_df = pd.DataFrame(cm, index=["Actual: si","Actual: no"], columns=["Pred: si","Pred: no"])
    print("Matriz de confusión:")
    print(cm_df)

print_metrics(y_test, y_pred_f, f"Mejor F1 (n_estimators={best_row_F['n_estimators']})")
print_metrics(y_test, y_pred_a, f"Mejor Accuracy (n_estimators={best_row_A['n_estimators']})")



=== Evaluando Random Forest con n_estimators=100 ===
Accuracy promedio CV: 0.7766 ± 0.0041
Accuracy en test: 0.7746
F1-score en test: 0.1392
Recall en test: 0.0823
Matriz de confusión:


ValueError: At least one label specified must be in y_true

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import algoritmos_clasificacion as t
from sklearn.model_selection import KFold

def crear_lags_concat(X, y, nd):
    """
    Genera variables lag (días previos) hasta 'nd' para cada columna de X.
    Devuelve X_lag e y_lag con las filas sincronizadas y sin NaN.
    """
    lags = {}
    for col in X.columns:
        for d in range(1, nd+1):
            lags[f"{col}_lag{d}"] = X[col].shift(d)

    X_lag = pd.concat([X, pd.DataFrame(lags)], axis=1)

    # Eliminar filas con NaN generadas por shift
    mask = ~X_lag.isna().any(axis=1)
    return X_lag[mask], y[mask]


# Hiperparámetros a probar con Bayes
m_values = [0.5, 1, 2, 5]
nd_values = [3] # [1, 2, 3]

resultados_bayes = {}

S = t.cargarDataset("data_dia")

X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto",
       "amplitud_carrasco","amplitud_salto",
       "dir_viento_carrasco_prom","int_viento_carrasco_prom","int_racha_viento_carrasco_prom",
       "dir_viento_salto_prom","int_viento_salto_prom","int_racha_viento_salto_prom",
       "vis_carrasco_prom","vis_salto_prom"]]

y = S["dia_lluvioso"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

for m in m_values:
    for nd in nd_values:
        print(f"\n=== Evaluando Naive Bayes con m={m}, nd={nd} ===")
        
        X_train_lag, y_train_lag = crear_lags_concat(X_train, y_train, nd)
        X_test_lag, y_test_lag = crear_lags_concat(X_test, y_test, nd)

        # Crear modelo
        model = t.NaiveBayesClassifier(m=m, nd=nd)
        
        # Validación cruzada en train
        cv_scores = cross_val_score(model, X_train_lag, y_train_lag, cv=kf)
        resultados_bayes[(m, nd)] = cv_scores
        
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train_lag, y_train_lag)
        y_pred = model.predict(X_test_lag)

        # Accuracy
        acc_test = accuracy_score(y_test, y_pred)
        # F1-score
        f1_test = f1_score(y_test, y_pred, average='binary', pos_label='si')
        # Recall
        recall_test = recall_score(y_test, y_pred, average='binary', pos_label='si')
        # Matriz de confusión
        cm = confusion_matrix(y_test, y_pred, labels=['si','no'])
        
        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)
# Resumen final de Naive Bayes para los distintos m y nd
print("\nRESUMEN NAIVE BAYES (Accuracy promedio CV ± desviación)")
for (m, nd), scores in resultados_bayes.items():
    print(f"m={m}, nd={nd} : {scores.mean():.4f} ± {scores.std():.4f}")


=== Evaluando Naive Bayes con m=0.5, nd=3 ===
Scores por fold: [np.float64(0.7641), np.float64(0.7595), np.float64(0.7555), np.float64(0.733), np.float64(0.7589)]
Accuracy promedio CV: 0.7542 ± 0.0109


ValueError: Found input variables with inconsistent numbers of samples: [4060, 4057]